In [76]:
import pandas as pd
import re
import requests

In [ ]:
df = pd.read_csv(
    "carros-fipe.csv",
    sep=";",
    encoding="latin1",
    na_values=["", "NULL", "null", "N/A"]
)
print(df.shape)
df.head()

df.info()

print(df.describe())



(50, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   registro_id   50 non-null     int64 
 1   marca_id      49 non-null     object
 2   modelo_id     49 non-null     object
 3   ano_modelo    49 non-null     object
 4   data_venda    50 non-null     object
 5   valor_venda   50 non-null     object
 6   taxa_servico  50 non-null     object
 7   valor_total   50 non-null     object
 8   km_rodados    50 non-null     object
 9   placa         47 non-null     object
 10  cliente       50 non-null     object
 11  cpf           49 non-null     object
 12  estado        50 non-null     object
dtypes: int64(1), object(12)
memory usage: 5.2+ KB
       registro_id
count     50.00000
mean      25.50000
std       14.57738
min        1.00000
25%       13.25000
50%       25.50000
75%       37.75000
max       50.00000


In [78]:
df_original = df.copy() 
if 'registro_id' in df.columns:
    df.drop(columns=['registro_id'], inplace=True)
print(df.shape)
df.head()

(50, 12)


,marca_id,modelo_id,ano_modelo,data_venda,valor_venda,taxa_servico,valor_total,km_rodados,placa,cliente,cpf,estado
0,59,5940,2011-3,2025-01-10,"R$ 88.629,00","1.200,00","89.829,00",120000,ABC1D23,Joao Silva,123.456.789-09,SP
1,59,5940,2021-3,10/01/2025,88629.00,1200.00,89829.00,120.000,ABC1D23,JOAO SILVA,12345678909,SP
2,59,5940,2020-3,01-10-2025,"88,629.00","1.200,00","89.829,00",120000km,ABC1D23,joao alberto silva,123-456-789-10,RJ
3,21,4828,2010-5,2025/02/15,"R$45.990,00","900,00","46.890,00",98000,DEF-4K56,Maria Souza Franco,987.654.321-00,SP
4,21,4828,2010-5,15/02/2025,45990,900,46890,98.000,DEF4K56,maria augusta souza,98765432100,SP


In [79]:
def limpar_numeros(val):
    if pd.isna(val): return None
    s = str(val).replace('R$', '').replace(' ', '').replace('km', '').strip()
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')
    try:
        return float(s)
    except:
        return None
df['valor_venda_num'] = df['valor_venda'].apply(limpar_numeros)
df['km_rodados_num'] = df['km_rodados'].apply(limpar_numeros)

print(df[['valor_venda_num', 'km_rodados_num']].describe())

       valor_venda_num  km_rodados_num
count        50.000000       48.000000
mean      55534.510320    59212.833333
std       24115.266257    45350.045958
min          88.629000       42.000000
25%       41422.500000      120.000000
50%       52300.000000    65000.000000
75%       73500.000000    98000.000000
max       88629.000000   120000.000000


In [80]:
def limpar_apenas_digitos(val):
    if pd.isna(val): return None
    return re.sub(r'\D', '', str(val))

def limpar_placa(val):
    if pd.isna(val): return None
    return re.sub(r'[^a-zA-Z0-9]', '', str(val))


In [90]:
df['cliente'] = df['cliente'].str.title().str.strip()
df['marca_id_clean'] = df['marca_id'].astype(str).str.extract(r'(\d+)')[0]
df['modelo_id_clean'] = df['modelo_id'].astype(str).str.extract(r'(\d+)')[0]
df['ano_modelo_clean'] = df['ano_modelo'].str.strip() 
df['placa_clean'] = df['placa'].apply(limpar_placa)
df['cpf_clean'] = df['cpf'].apply(limpar_apenas_digitos)
df['valor_venda_clean'] = df['valor_venda'].apply(limpar_numeros)
df['taxa_servico_clean'] = df['taxa_servico'].apply(limpar_numeros)
print(df[[ 'cliente', 'marca_id_clean', 'modelo_id_clean', 'ano_modelo_clean', 'placa_clean', 'cpf_clean', 'valor_venda_clean', 'taxa_servico_clean']].head())

               cliente marca_id_clean modelo_id_clean ano_modelo_clean  \
0           Joao Silva             59            5940           2011-3   
1           Joao Silva             59            5940           2021-3   
2   Joao Alberto Silva             59            5940           2020-3   
3   Maria Souza Franco             21            4828           2010-5   
4  Maria Augusta Souza             21            4828           2010-5   

  placa_clean    cpf_clean  valor_venda_clean  taxa_servico_clean  
0     ABC1D23  12345678909          88629.000              1200.0  
1     ABC1D23  12345678909          88629.000              1200.0  
2     ABC1D23  12345678910             88.629              1200.0  
3     DEF4K56  98765432100          45990.000               900.0  
4     DEF4K56  98765432100          45990.000               900.0  


In [91]:
is_valid = (
    df['placa_clean'].notna() & 
    df['cpf_clean'].notna() & 
    df['marca_id_clean'].notna() &
    df['modelo_id_clean'].notna() &
    df['ano_modelo_clean'].notna()
)

df_descarte = df_original[~is_valid].copy()
df_working = df[is_valid].copy()
print(f"Registros válidos: {len(df_working)}")
print(f"Registros para descarte: {len(df_descarte)}")


Registros válidos: 42
Registros para descarte: 8


In [101]:
def consultar_fipe(row):
    url = f"https://parallelum.com.br/fipe/api/v1/carros/marcas/{row['marca_id_clean']}/modelos/{row['modelo_id_clean']}/anos/{row['ano_modelo_clean']}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()
    except:
        pass
    return None
df_working['fipe_json'] = df_working.apply(consultar_fipe, axis=1)



In [93]:
is_endpoint_ok = df_working['fipe_json'].notna()
df_descarte_endpoint = df_working[~is_endpoint_ok].copy()
df_final = df_working[is_endpoint_ok].copy()
print(f"Registros com dados FIPE: {len(df_final)}")
print(f"Registros descartados por falha na consulta FIPE: {len(df_descarte_endpoint)}")


Registros com dados FIPE: 10
Registros descartados por falha na consulta FIPE: 32


In [94]:
def formatar_valor_fipe(json_data):
    valor_str = json_data.get('Valor', '0')
    return valor_str.replace('R$', '').strip()

df_final['preco_fipe'] = df_final['fipe_json'].apply(formatar_valor_fipe)
df_final['marca'] = df_final['fipe_json'].apply(lambda x: x.get('Marca', ''))
print(df_final[['preco_fipe', 'marca']].head())

   preco_fipe            marca
0   72.434,00  VW - VolksWagen
1  149.786,00  VW - VolksWagen
2  141.217,00  VW - VolksWagen
3   25.693,00             Fiat
4   25.693,00             Fiat


In [95]:
df_final['preco_fipe_num'] = df_final['preco_fipe'].apply(lambda x: float(x.replace('.', '').replace(',', '.')))
df_final['lucro_num'] = df_final['preco_fipe_num'] - df_final['valor_venda_clean']
df_final['lucro'] = df_final['lucro_num'].apply(lambda x: "{:.2f}".format(x).replace('.', ','))
print(df_final[['preco_fipe', 'valor_venda_clean', 'lucro']].head())

   preco_fipe  valor_venda_clean      lucro
0   72.434,00          88629.000  -16195,00
1  149.786,00          88629.000   61157,00
2  141.217,00             88.629  141128,37
3   25.693,00          45990.000  -20297,00
4   25.693,00          45990.000  -20297,00


In [96]:
df_final['valor_total'] = (df_final['valor_venda_clean'] + df_final['taxa_servico_clean']).apply(lambda x: "{:.2f}".format(x).replace('.', ','))
df_final['valor_venda'] = df_final['valor_venda_clean'].apply(lambda x: "{:.2f}".format(x).replace('.', ','))
df_final['taxa_servico'] = df_final['taxa_servico_clean'].apply(lambda x: "{:.2f}".format(x).replace('.', ','))
df_final['km_rodados'] = df_final['km_rodados'].apply(limpar_apenas_digitos)
print(df_final[['valor_venda', 'taxa_servico', 'valor_total', 'km_rodados']].head())

  valor_venda taxa_servico valor_total km_rodados
0    88629,00      1200,00    89829,00     120000
1    88629,00      1200,00    89829,00     120000
2       88,63      1200,00     1288,63     120000
3    45990,00       900,00    46890,00      98000
4    45990,00       900,00    46890,00      98000


In [97]:
colunas_saida = [
    'marca_id', 'modelo_id', 'ano_modelo', 'data_venda', 'valor_venda', 
    'taxa_servico', 'valor_total', 'km_rodados', 'placa', 'cliente', 
    'cpf', 'estado', 'preco_fipe', 'marca', 'lucro'
]
print(df_final[colunas_saida].head())

  marca_id modelo_id ano_modelo  data_venda valor_venda taxa_servico  \
0       59      5940   2011-3    2025-01-10    88629,00      1200,00   
1       59      5940     2021-3  10/01/2025    88629,00      1200,00   
2       59      5940     2020-3  01-10-2025       88,63      1200,00   
3       21      4828     2010-5  2025/02/15    45990,00       900,00   
4       21      4828     2010-5  15/02/2025    45990,00       900,00   

  valor_total km_rodados     placa              cliente             cpf  \
0    89829,00     120000   ABC1D23           Joao Silva  123.456.789-09   
1    89829,00     120000   ABC1D23           Joao Silva     12345678909   
2     1288,63     120000   ABC1D23   Joao Alberto Silva  123-456-789-10   
3    46890,00      98000  DEF-4K56   Maria Souza Franco  987.654.321-00   
4    46890,00      98000   DEF4K56  Maria Augusta Souza     98765432100   

  estado  preco_fipe            marca      lucro  
0     SP   72.434,00  VW - VolksWagen  -16195,00  
1     SP  149.

In [98]:
df_final[colunas_saida].to_csv('carros-fipe-saida.csv', index=False, sep=';')
df_descarte.to_csv('carros-fipe-descarte.csv', index=False, sep=';')
df_descarte_endpoint.to_csv('carros-fipe-descarte-endpoint.csv', index=False, sep=';')

In [99]:
df_final['ano_ext'] = df_final['ano_modelo'].str[:4]
pivot = pd.pivot_table(
    df_final, 
    values='valor_venda_clean', 
    index='marca', 
    columns='ano_ext', 
    aggfunc='sum', 
    fill_value=0
)
pivot.to_csv('total_por_marca.csv', sep=';')

print("Processamento concluído com sucesso!")

Processamento concluído com sucesso!
